In [3]:
import pandas as pd

# Load the original dataset without modifying it
df_original = pd.read_csv("../data/H2.csv")

# Create a separate version with exact duplicate rows removed
df_non_duplicated = df_original.drop_duplicates().copy()

# Compare their sizes
print("Original dataset shape:", df_original.shape)
print("Deduplicated dataset shape:", df_non_duplicated.shape)
print("Duplicate occurrences removed:",
      len(df_original) - len(df_non_duplicated))


Original dataset shape: (79330, 31)
Deduplicated dataset shape: (53428, 31)
Duplicate occurrences removed: 25902


In [4]:
from sklearn.impute import SimpleImputer # fills missing values using a chosen rule.
from sklearn.compose import ColumnTransformer # applies different rules to numeric and categorical columns.
from sklearn.pipeline import Pipeline # connects preprocessing and model training into one workflow.
from sklearn.preprocessing import OneHotEncoder # converts categories into numeric features.
from sklearn.preprocessing import StandardScaler # puts numeric features on comparable scales when needed

In [5]:
# Columns that would cause information leakage in classification model
leakage_columns = [
    "ReservationStatus",
    "ReservationStatusDate"
]

# Separate features (X) and target (y)
X_original = df_original.drop(
    columns=["IsCanceled"] + leakage_columns
)
y_original = df_original["IsCanceled"]

X_non_duplicated = df_non_duplicated.drop(
    columns=["IsCanceled"] + leakage_columns
)
y_non_duplicated = df_non_duplicated["IsCanceled"]

# Check the resulting dimensions
print("Original features:", X_original.shape)
print("Original target:", y_original.shape)

print("Deduplicated features:", X_non_duplicated.shape)
print("Deduplicated target:", y_non_duplicated.shape)

Original features: (79330, 28)
Original target: (79330,)
Deduplicated features: (53428, 28)
Deduplicated target: (53428,)


In [6]:

from sklearn.model_selection import GroupShuffleSplit

# Identify identical feature-and-target records
groups_original = pd.util.hash_pandas_object(
    pd.concat([X_original, y_original.rename("IsCanceled")], axis=1),
    index=False
).to_numpy()

groups_non_duplicated = pd.util.hash_pandas_object(
    pd.concat([
        X_non_duplicated,
        y_non_duplicated.rename("IsCanceled")
    ], axis=1),
    index=False
).to_numpy()


# Split the original dataset
splitter_original = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)

train_idx_original, test_idx_original = next(
    splitter_original.split(
        X_original, y_original, groups_original
    )
)

X_train_original = X_original.iloc[train_idx_original]
X_test_original = X_original.iloc[test_idx_original]
y_train_original = y_original.iloc[train_idx_original]
y_test_original = y_original.iloc[test_idx_original]


# Split the dataset without duplicates
splitter_non_duplicated = GroupShuffleSplit(
    n_splits=1, test_size=0.2, random_state=42
)

train_idx_non_duplicated, test_idx_non_duplicated = next(
    splitter_non_duplicated.split(
        X_non_duplicated,
        y_non_duplicated,
        groups_non_duplicated
    )
)

X_train_non_duplicated = X_non_duplicated.iloc[
    train_idx_non_duplicated
]
X_test_non_duplicated = X_non_duplicated.iloc[
    test_idx_non_duplicated
]
y_train_non_duplicated = y_non_duplicated.iloc[
    train_idx_non_duplicated
]
y_test_non_duplicated = y_non_duplicated.iloc[
    test_idx_non_duplicated
]


# Display split sizes
print("ORIGINAL DATASET")
print("Training rows:", len(X_train_original))
print("Testing rows:", len(X_test_original))

print("\nNON-DUPLICATED DATASET")
print("Training rows:", len(X_train_non_duplicated))
print("Testing rows:", len(X_test_non_duplicated))


# Check for group overlap
print(
    "\nOriginal groups overlap:",
    bool(
        set(groups_original[train_idx_original])
        & set(groups_original[test_idx_original])
    )
)

print(
    "Non-duplicated groups overlap:",
    bool(
        set(groups_non_duplicated[train_idx_non_duplicated])
        & set(groups_non_duplicated[test_idx_non_duplicated])
    )
)


ORIGINAL DATASET
Training rows: 63483
Testing rows: 15847

NON-DUPLICATED DATASET
Training rows: 42747
Testing rows: 10681

Original groups overlap: False
Non-duplicated groups overlap: False


In [7]:
# Identify numeric and categorical columns from the training data
numeric_features = X_train_original.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train_original.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Number of numeric features:", len(numeric_features))
print("Numeric features:", numeric_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Categorical features:", categorical_features)

Number of numeric features: 17
Numeric features: ['LeadTime', 'ArrivalDateYear', 'ArrivalDateWeekNumber', 'ArrivalDateDayOfMonth', 'StaysInWeekendNights', 'StaysInWeekNights', 'Adults', 'Children', 'Babies', 'IsRepeatedGuest', 'PreviousCancellations', 'PreviousBookingsNotCanceled', 'BookingChanges', 'DaysInWaitingList', 'ADR', 'RequiredCarParkingSpaces', 'TotalOfSpecialRequests']

Number of categorical features: 11
Categorical features: ['ArrivalDateMonth', 'Meal', 'Country', 'MarketSegment', 'DistributionChannel', 'ReservedRoomType', 'AssignedRoomType', 'DepositType', 'Agent', 'Company', 'CustomerType']


In [8]:
# Low-cardinality categorical features
# These will be one-hot encoded directly.
# ArrivalDateMonth here is with low caridality even tho it has more than 10 crdial values but because it is managable to be 12 and they are important so it is manegable
low_cardinality_features = [
    "AssignedRoomType",
    "MarketSegment",
    "ReservedRoomType",
    "DistributionChannel",
    "Meal",
    "CustomerType",
    "DepositType",
    "ArrivalDateMonth"
]

# High-cardinality categorical features
# Rare categories will be grouped into "Other".

high_cardinality_features = [
    "Agent",
    "Company",
    "Country"
]

print("Low-cardinality features:", low_cardinality_features)
print("High-cardinality features:", high_cardinality_features)

Low-cardinality features: ['AssignedRoomType', 'MarketSegment', 'ReservedRoomType', 'DistributionChannel', 'Meal', 'CustomerType', 'DepositType', 'ArrivalDateMonth']
High-cardinality features: ['Agent', 'Company', 'Country']


In [12]:
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_frequency=0.01): ## if this value is less than 1% of the vlues in the data then it is considered rare and put with "other" category
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        self.frequent_categories_ = {}

        X = X.copy()

        for column in X.columns:
            # Convert non-missing values to strings
            # while preserving missing values
            X[column] = X[column].astype(object).where(
                X[column].isna(),
                X[column].astype(str)
            )

            # Calculate the proportion of each category
            frequencies = X[column].value_counts(
                normalize=True,
                dropna=True
            )

            # Keep categories appearing in at least 1% of training rows
            self.frequent_categories_[column] = set(
                frequencies[
                    frequencies >= self.min_frequency
                ].index
            )

        return self

    def transform(self, X):
        X = X.copy()

        for column in X.columns:
            # Apply the same conversion used during fitting
            X[column] = X[column].astype(object).where(
                X[column].isna(),
                X[column].astype(str)
            )

            frequent = self.frequent_categories_[column]

            # Replace rare categories with "Other"
            # Keep missing values for the imputer
            X[column] = X[column].where(
                X[column].isna() | X[column].isin(frequent),
                "Other"
            )

        return X

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Numeric features:
# Fill missing values with the median, then scale the values.
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 2. Low-cardinality categorical features:
# Fill missing values with the most frequent category,
# then one-hot encode the categories.
low_cardinality_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 3. High-cardinality categorical features:
# Group rare categories, fill missing values, then one-hot encode.
high_cardinality_transformer = Pipeline(steps=[
    ("rare_grouper", RareCategoryGrouper(min_frequency=0.01)),
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the three preprocessing pipelines
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("low_cardinality", low_cardinality_transformer,
     low_cardinality_features),
    ("high_cardinality", high_cardinality_transformer,
     high_cardinality_features)
])

print("Cardinality-aware preprocessing pipeline created.")

Cardinality-aware preprocessing pipeline created.


In [11]:
# Learn preprocessing rules from the original training data
X_train_original_transformed = preprocessor.fit_transform(
    X_train_original
)

# Apply those same rules to the original test data
X_test_original_transformed = preprocessor.transform(
    X_test_original
)

print("Original dataset preprocessing complete.")
print("Transformed training shape:", X_train_original_transformed.shape)
print("Transformed testing shape:", X_test_original_transformed.shape)

Original dataset preprocessing complete.
Transformed training shape: (63483, 103)
Transformed testing shape: (15847, 103)
